In [ ]:
from huggingface_hub import notebook_login

# This will create a popup for you to paste your token
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


malaria


"The patient presents with high-grade fever that occurs in cycles every 48 hours. Each episode begins with intense shivering and chills, followed by a period of extreme heat and sweating. They are also experiencing a moderate headache, nausea, and significant fatigue. The patient recently returned from a tropical region where malaria is endemic. There is no cough, chest pain, or shortness of breath."

In [ ]:
user_input = "The patient presents with high-grade fever that occurs in cycles every 48 hours. Each episode begins with intense shivering and chills, followed by a period of extreme heat and sweating. They are also experiencing a moderate headache, nausea, and significant fatigue. The patient recently returned from a tropical region where malaria is endemic. There is no cough, chest pain, or shortness of breath."

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json
import re

# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# --- 1. SETUP ---
# Use the newer 1.5-4b-it for better performance and smaller memory footprint
MODEL_ID = "google/medgemma-1.5-4b-it"

# Ensure your token is set
import os
os.environ["HF_TOKEN"] = "[xxxxxxxxxxxxxxxxxxxxxxxxxxxxx]"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)


# Define the specific model you want to use
model_id = "google/medgemma-1.5-4b-it"
from transformers import AutoProcessor

# Initialize the processor for MedGemma 1.5
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
# --- 2. THE DIAGNOSTIC FUNCTION ---
# --- UPDATED DIAGNOSTIC FUNCTION (SINGLE OUTPUT) ---
def get_clean_diagnosis(symptoms):
    # Updated prompt to focus on the single most likely condition
    # This prevents the model from generating a Top 3 list.
    prompt = f"""
    Task: Identify the single most likely medical diagnosis for the symptoms below.
    Format: (Disease Name Confidence%)

    Example 1:
    Symptoms: High fever, chills, and sweats.
    Output: (Malaria 95%)

    Example 2:
    Symptoms: Persistent rash and itchy skin.
    Output: (Psoriasis 88%)

    Your Turn:
    Symptoms: {symptoms}
    Output:"""

    inputs = processor(text=prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=25, # Reduced to keep the output focused on one item
            temperature=0.1,   # Low temperature for high precision
            do_sample=False
        )

    response = processor.batch_decode(generated_ids, skip_special_tokens=True)

    # Extracting the single output line
    final_output = response[0].split("Output:")[-1].strip()
    return final_output

# --- TEST RUN ---
result = get_clean_diagnosis(user_input)
print(result)
# Expected Output: (Pneumonia 88%, Bronchial Asthma 8%, Common Cold 4%)

Using device: cuda


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


(Malaria 99%)
    ```

    Your Turn:
    Symptoms: The patient presents with high-


In [ ]:
import re
def parse_diagnosis_to_mapping(output_string):
    # 1. Clean the string: Remove backticks, parentheses, and newlines
    clean_str = output_string.replace('`', '').replace('(', '').replace(')', '').strip()

    # 2. Regex: Find (Text) followed by (Digits) and a (%)
    # This works regardless of how many backticks the LLM added
    matches = re.findall(r'([a-zA-Z\s]+)\s+(\d+)\s*%', clean_str)

    # 3. Create the mapping (lowercase keys for consistency)
    mapping = {disease.strip().lower(): int(score) for disease, score in matches}

    return mapping
mapping=parse_diagnosis_to_mapping(result)

In [ ]:
!pip install chembl_webresource_client pubchempy rdkit selfies

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00


In [ ]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Descriptors

# --- 0. DATA-LEVEL GUARDRAIL ---
def is_gan_friendly(smi):
    """Filters out gases, massive proteins, and overly complex 3D structures."""
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return False

        # 1. Reject Gases/Tiny Fragments (Fixes the 'O=O' Asthma problem)
        if mol.GetNumHeavyAtoms() < 8:
            return False

        # 2. Reject Massive Molecules (Keeps it within GAN memory capacity)
        if Descriptors.MolWt(mol) > 600:
            return False

        # 3. Reject High 3D Complexity (Fixes the Pneumonia Vitamin D problem)
        # We limit the number of stereocenters (@) to keep the 'grammar' simple for the GAN
        if smi.count('@') > 3:
            return False

        return True
    except:
        return False

# --- 1. STRICT SEARCH ENGINE (With Data-Level Filtering) ---
try:
    from chembl_webresource_client.new_client import new_client
except Exception as e:
    print(f"⚠️ ChEMBL Library Error: {e}")
    new_client = None

def get_smiles_strict(disease_name):
    """Fetches high-quality SMILES. Halts the pipeline if no GAN-friendly data is found."""

    # --- 1. PRIMARY: CHEMBL ---
    if new_client:
        try:
            print(f"📡 Querying ChEMBL for {disease_name.upper()}...")
            indications = new_client.drug_indication.filter(mesh_heading__icontains=disease_name)

            # Loop through indications to find a GAN-friendly one
            for entry in indications[:10]:  # Check top 10 results
                chembl_id = entry['molecule_chembl_id']
                mol_data = new_client.molecule.filter(molecule_chembl_id=chembl_id).only(['molecule_structures'])

                if mol_data and mol_data[0].get('molecule_structures'):
                    smi = mol_data[0]['molecule_structures']['canonical_smiles']
                    if is_gan_friendly(smi):
                        print(f"✅ GAN-Friendly Data Found (ChEMBL): {smi}")
                        return smi
                    else:
                        print(f"⏩ Skipping {smi} (Too complex or too small)")
        except Exception as e:
            print(f"⚠️ ChEMBL API error: {e}")

    # --- 2. SECONDARY: PUBCHEM ---
    try:
        print(f"📡 Querying PubChem for {disease_name.upper()}...")
        results = pcp.get_compounds(disease_name, 'name')

        # Loop through PubChem results to find a GAN-friendly one
        for res in results[:5]:
            smi = res.canonical_smiles
            if is_gan_friendly(smi):
                print(f"✅ GAN-Friendly Data Found (PubChem): {smi}")
                return smi
            else:
                print(f"⏩ Skipping {smi} (Failed Guardrail)")
    except Exception as e:
        print(f"❌ PubChem API Error: {e}")

    # --- 3. THE STRICT HALT ---
    raise RuntimeError(
        f"\n🛑 DATA-LEVEL FAILURE: Found no GAN-friendly molecules for '{disease_name}'. "
        f"The database returned only gases, proteins, or complex vitamins that the GAN cannot process. "
        f"Pipeline halted to prevent low-quality training."
    )

# --- REPLACING THE HYBRID WELDER ---
def select_best_lead_molecule(mapping):
    if not mapping:
        raise ValueError("🛑 CRITICAL FAILURE: MedGemma provided an empty diagnosis mapping.")

    best_disease = max(mapping, key=mapping.get)
    confidence = mapping[best_disease]

    print(f"\n🎯 Top Diagnosis: {best_disease.upper()} ({confidence}%)")
    lead_smi = get_smiles_strict(best_disease)

    print(f"🏠 Lead Molecule Secured: {lead_smi}")
    return lead_smi

# --- UPDATED EXECUTION ---
final_smi = select_best_lead_molecule(mapping)


🎯 Top Diagnosis: MALARIA (99%)
📡 Querying ChEMBL for MALARIA...
✅ GAN-Friendly Data Found (ChEMBL): CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(C(=O)N[C@@H](CCC(=O)O)C(=O)O)cc1
🏠 Lead Molecule Secured: CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(C(=O)N[C@@H](CCC(=O)O)C(=O)O)cc1


In [ ]:
!pip install torch rdkit pypi selfies pandas

  Preparing metadata (setup.py) ... done
  Created wheel for pypi: filename=pypi-2.1-py3-none-any.whl size=1334 sha256=321933b495d59eaf0fd8e39780d8bfcb1611c9d125eddc5a3b10a4355eaa4c40
  Stored in directory: /root/.cache/pip/wheels/28/4c/49/00cdce1e7a68a48810e9203391f80f4c7344a5e4ad9d4d6649
Successfully built pypi


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import pandas as pd
import numpy as np
import selfies as sf
import random
import math
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import QED, Crippen, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

# -------------------------------
# SEED MOLECULE
# -------------------------------
# Cell 12 Update
seed_smiles = final_smi # This is now the single lead drug
seed_mol = Chem.MolFromSmiles(seed_smiles)

# Extract the Murcko Scaffold of the real drug
seed_scaffold = MurckoScaffold.GetScaffoldForMol(seed_mol)
scaffold_smiles = Chem.MolToSmiles(seed_scaffold)

print(f"Using Validated Drug Scaffold: {scaffold_smiles}")

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using Validated Drug Scaffold: c1ccc(NCc2cnc3ncncc3n2)cc1
Using device: cuda


In [ ]:
# Cell 3: Download and load the dataset (with checkpointing a subset)
import os
from google.colab import drive

# --- 1. Mount Google Drive ---
drive.mount('/content/drive')

# --- 2. Define the path for your checkpoint file ---
checkpoint_folder = '/content/drive/My Drive/molgen_project_(1)/'
checkpoint_file = os.path.join(checkpoint_folder, 'selfies_processed.txt')
os.makedirs(checkpoint_folder, exist_ok=True)

# --- DEFINE YOUR DESIRED DATASET SIZE ---
# You can change this number to experiment with different sizes (e.g., 500000)
DATASET_SUBSET_SIZE = 25000

# --- 3. Check if the processed SELFIES file already exists ---
if os.path.exists(checkpoint_file):
    print(f"✅ Checkpoint found! Loading pre-processed SELFIES from: {checkpoint_file}")
    with open(checkpoint_file, 'r') as f:
        # Load the full list from the file (this is fast)
        full_selfies_list = [line.strip() for line in f]

    # --- CHANGE: Take only the first 250,000 molecules from the loaded list ---
    selfies_list = full_selfies_list[:DATASET_SUBSET_SIZE]
    print(f"Successfully loaded {len(full_selfies_list)} molecules, using the first {len(selfies_list)} for training.")

else:
    # This block will only run if you've never created the checkpoint before
    print("⏳ No checkpoint found. Processing SMILES to SELFIES for the first time...")
    url = "https://media.githubusercontent.com/media/molecularsets/moses/master/data/dataset_v1.csv"
    df = pd.read_csv(url)
    smiles_list = df['SMILES'].tolist()
    full_selfies_list = [sf.encoder(smi) for smi in tqdm(smiles_list, desc="Encoding SELFIES")]

    print(f"💾 Saving all {len(full_selfies_list)} processed SELFIES to checkpoint for future use.")
    with open(checkpoint_file, 'w') as f:
        for selfies_string in full_selfies_list:
            f.write(f"{selfies_string}\n")
    print("Checkpoint saved successfully.")

    # Use the subset for the current session
    selfies_list = full_selfies_list[:DATASET_SUBSET_SIZE]
    print(f"Using the first {len(selfies_list)} molecules for training.")

    # Save the result to your Google Drive for next time
    print(f"💾 Saving processed SELFIES to checkpoint: {checkpoint_file}")
    with open(checkpoint_file, 'w') as f:
        for selfies_string in selfies_list:
            f.write(f"{selfies_string}\n")
    print("Checkpoint saved successfully.")


# Cell 4: Build Vocabulary
def build_vocab(selfies_list):
    """Builds a vocabulary and token mappings from a list of SELFIES strings."""
    tokens = set()
    for s in selfies_list:
        tokens.update(sf.split_selfies(s))

    vocab = sorted(list(tokens))
    vocab.insert(0, '<pad>') # Padding token
    vocab.insert(1, '<sos>') # Start of sequence
    vocab.insert(2, '<eos>') # End of sequence

    token_to_int = {token: i for i, token in enumerate(vocab)}
    int_to_token = {i: token for i, token in enumerate(vocab)}

    return vocab, token_to_int, int_to_token

vocab, token_to_int, int_to_token = build_vocab(selfies_list)
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

# Cell 5: Create a PyTorch Dataset
class SELFIESDataset(Dataset):
    """PyTorch Dataset for SELFIES sequences."""
    def __init__(self, selfies_list, token_to_int, max_len=128):
        self.selfies_list = selfies_list
        self.token_to_int = token_to_int
        self.max_len = max_len
        self.pad_token_id = token_to_int['<pad>']
        self.sos_token_id = token_to_int['<sos>']
        self.eos_token_id = token_to_int['<eos>']

    def __len__(self):
        return len(self.selfies_list)

    def __getitem__(self, idx):
        selfies = sf.split_selfies(self.selfies_list[idx])
        tokens = ['<sos>'] + list(selfies) + ['<eos>']

        # Truncate if longer than max_len
        if len(tokens) > self.max_len:
            tokens = tokens[:self.max_len]

        # Convert to integers
        token_ids = [self.token_to_int.get(token, self.pad_token_id) for token in tokens]

        # Pad sequence
        padding_len = self.max_len - len(token_ids)
        token_ids += [self.pad_token_id] * padding_len

        return torch.tensor(token_ids, dtype=torch.long)

# Hyperparameters
MAX_LEN = 128
BATCH_SIZE = 16

dataset = SELFIESDataset(selfies_list, token_to_int, max_len=MAX_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

Mounted at /content/drive
✅ Checkpoint found! Loading pre-processed SELFIES from: /content/drive/My Drive/molgen_project_(1)/selfies_processed.txt
Successfully loaded 25000 molecules, using the first 25000 for training.
Vocabulary size: 29


In [ ]:
# Cell 6: Transformer Generator [cite: 66, 95]
class TransformerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=512, max_seq_length=128):
        super(TransformerGenerator, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_seq_length, d_model))
        self.transformer = nn.Transformer(d_model, nhead, num_encoder_layers,
                                           num_decoder_layers, dim_feedforward, batch_first=True)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt):
        src_emb = self.embedding(src) * math.sqrt(self.d_model) + self.positional_encoding[:, :src.size(1), :]
        tgt_emb = self.embedding(tgt) * math.sqrt(self.d_model) + self.positional_encoding[:, :tgt.size(1), :]

        # Create masks
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(device)

        output = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask)
        return self.fc_out(output)

# Modified CNNDiscriminator to accept pre-embedded inputs
class CNNDiscriminator(nn.Module):
    def __init__(self, vocab_size, d_model=256, max_seq_length=128):
        super(CNNDiscriminator, self).__init__()
        # The embedding layer is still part of the model's parameters
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, 100, (k, d_model)) for k in [2, 3, 4, 5]
        ])
        self.fc = nn.Linear(400, 1)

    def forward(self, embedded_x):
        # The forward pass now starts with the already embedded input
        # The input should have shape: (batch_size, seq_len, d_model)
        embedded = embedded_x.unsqueeze(1) # (batch_size, 1, seq_len, d_model)
        conved = [F.relu(conv(embedded)).squeeze(3) for conv in self.convs]
        pooled = [F.max_pool1d(c, c.size(2)).squeeze(2) for c in conved]
        cat = torch.cat(pooled, 1)
        return self.fc(cat)

In [ ]:
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
import random

activity = new_client.activity
target = new_client.target




def fetch_disease_molecules(disease_name, max_mols=2000, ic50_threshold=1000):

    organisms = DISEASE_TO_ORGANISM.get(disease_name.lower(), [disease_name])

    smiles_list = []

    print(f"\n🔎 Fetching molecules for: {disease_name}")

    for organism in organisms:

        print(f"   🎯 Searching organism: {organism}")

        # STEP 1: Get all targets for that organism
        organism_targets = target.filter(organism__icontains=organism)

        target_ids = [t['target_chembl_id'] for t in organism_targets]

        print(f"      Found {len(target_ids)} targets")

        # STEP 2: Fetch activity for each target
        for t_id in target_ids[:20]:   # limit to avoid API overload

            results = activity.filter(
                target_chembl_id=t_id,
                standard_type="IC50",
                standard_units="nM",
                standard_value__lte=ic50_threshold
            ).only(['canonical_smiles'])[:max_mols]

            for r in results:
                smi = r.get("canonical_smiles")
                if smi:
                    mol = Chem.MolFromSmiles(smi)
                    if mol:
                        smiles_list.append(smi)

    print(f"   ✅ Retrieved {len(smiles_list)} molecules")
    return list(set(smiles_list))


def clean_smiles_list(smiles_list, min_mw=200, max_mw=600, min_heavy_atoms=15):
    cleaned = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mw = Descriptors.MolWt(mol)
            if min_mw <= mw <= max_mw and mol.GetNumHeavyAtoms() >= min_heavy_atoms:
                cleaned.append(smi)
    return cleaned


# ==============================================================================
# --- 1. IDENTIFY THE SINGLE BEST LEAD ---
# ==============================================================================
# Instead of mixing 3 diseases, we pick the top one from MedGemma's output
best_disease = max(mapping, key=mapping.get)
final_smi = get_smiles_strict(best_disease)

# ==============================================================================
# --- 2. TRANSLATE SMILES TO SELFIES FOR FINE-TUNING ---
# ==============================================================================
import selfies as sf
from rdkit import Chem
from torch.utils.data import DataLoader

target_smiles = [final_smi] * 500
print(f"🎯 Creating focused fine-tune set for: {final_smi}")

selfies_dataset = []

for smi in target_smiles:
    try:
        # Step A: Load the raw SMILES from ChEMBL/PubChem
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue

        # Step B: Flatten the molecule (Remove 3D data the AI vocab doesn't know)
        Chem.RemoveStereochemistry(mol)

        # Step C: Generate a clean, flat SMILES string
        flat_smi = Chem.MolToSmiles(mol, canonical=True)

        # Step D: CONVERT SMILES TO SELFIES (The AI's native language)
        s = sf.encoder(flat_smi)

        # Step E: Verify the AI's dictionary knows all the SELFIES tokens
        tokens = list(sf.split_selfies(s))
        if all(tok in token_to_int for tok in tokens):
            selfies_dataset.append(s)
        else:
            missing = [tok for tok in tokens if tok not in token_to_int]
            print(f"⚠️ Translation failed. Missing AI tokens: {set(missing)}")
            break # Stop the loop so it doesn't spam the output

    except Exception as e:
        continue

print(f"✅ Valid Lead SELFIES successfully translated: {len(selfies_dataset)}")

# ==============================================================================
# --- 3. CREATE DATA LOADER (With Safety Catch) ---
# ==============================================================================
if len(selfies_dataset) == 0:
    raise RuntimeError(
        "🛑 FATAL: The translation failed because the drug contains atoms completely "
        "unknown to your AI's vocabulary. Please try a different lead drug."
    )

fine_tune_dataset = SELFIESDataset(
    selfies_dataset,
    token_to_int,
    max_len=MAX_LEN
)

fine_tune_loader = DataLoader(
    fine_tune_dataset,
    batch_size=min(BATCH_SIZE, len(selfies_dataset)),
    shuffle=True
)

print("✅ DataLoader ready for Fine-Tuning!")

📡 Querying ChEMBL for MALARIA...
✅ GAN-Friendly Data Found (ChEMBL): CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(C(=O)N[C@@H](CCC(=O)O)C(=O)O)cc1
🎯 Creating focused fine-tune set for: CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(C(=O)N[C@@H](CCC(=O)O)C(=O)O)cc1
✅ Valid Lead SELFIES successfully translated: 500
✅ DataLoader ready for Fine-Tuning!


In [ ]:
# Define Hyperparameters for GAN Training
D_MODEL = 128     # Dimension of the model
LR = 1e-4         # Learning rate
LAMBDA_GP = 10.0  # Gradient penalty coefficient
N_EPOCHS = 50      # Number of epochs for GAN training

In [ ]:
# Cell 8: WGAN-GP Gradient Penalty Calculation (Your improved version is kept)
def compute_gradient_penalty(discriminator, real_samples_embedded, fake_samples_embedded):
    """Calculates the gradient penalty for WGAN-GP on embedded samples."""
    alpha = torch.randn(real_samples_embedded.size(0), 1, 1).to(device)
    alpha = alpha.expand_as(real_samples_embedded)

    interpolates = (alpha * real_samples_embedded + ((1 - alpha) * fake_samples_embedded)).requires_grad_(True)
    d_interpolates = discriminator(interpolates)

    fake = torch.autograd.Variable(torch.Tensor(real_samples_embedded.shape[0], 1).fill_(1.0), requires_grad=False).to(device)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

# Cell 9: Corrected Training Loop with Checkpointing
import os
from google.colab import drive

# --- 1. Define Checkpoint Paths (using the same folder as before) ---
# Ensure your drive is mounted from Cell 3. If not, re-run that cell.
checkpoint_folder = '/content/drive/My Drive/molgen_project_(1)/'
wgan_checkpoint_path = os.path.join(checkpoint_folder, 'wgan_checkpoint.pth')


# --- 2. Initialize Models and Optimizers ---
# These must be defined before we can load a state into them.

generator = TransformerGenerator(vocab_size, d_model=D_MODEL, max_seq_length=MAX_LEN).to(device)
discriminator = CNNDiscriminator(vocab_size, d_model=D_MODEL, max_seq_length=MAX_LEN).to(device)
optimizer_g = optim.Adam(generator.parameters(), lr=LR, betas=(0.5, 0.9))
optimizer_d = optim.Adam(discriminator.parameters(), lr=LR, betas=(0.5, 0.9))
criterion_g = nn.CrossEntropyLoss(ignore_index=token_to_int['<pad>'])


# --- 3. Load from Checkpoint if it Exists ---
start_epoch = 0
if os.path.exists(wgan_checkpoint_path):
    print(f"✅ WGAN checkpoint found! Loading state from: {wgan_checkpoint_path}")
    checkpoint = torch.load(wgan_checkpoint_path)

    generator.load_state_dict(checkpoint['generator_state_dict'])
    discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
    optimizer_g.load_state_dict(checkpoint['optimizer_g_state_dict'])
    optimizer_d.load_state_dict(checkpoint['optimizer_d_state_dict'])
    start_epoch = checkpoint['epoch'] + 1 # Start from the next epoch

    print(f"Resuming training from epoch {start_epoch}")
else:
    print("⏳ No WGAN checkpoint found. Starting training from scratch.")


# --- 4. Main Training Loop ---
print("Starting WGAN-GP Training...")
# The loop now starts from the last saved epoch (or 0 if starting fresh)
for epoch in range(start_epoch, N_EPOCHS):
    # (The inner loop logic for training one epoch remains the same)
    for i, real_seq in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{N_EPOCHS}")):
        real_seq = real_seq.to(device)
        # ... (rest of the training logic from the previous corrected Cell 9) ...
        # --- (All the code for training the Discriminator and Generator) ---
        # --------------------
        #  Train Discriminator
        # --------------------
        optimizer_d.zero_grad()
        noise_input = torch.randint(0, vocab_size, real_seq.shape, device=device)
        fake_output_logits = generator(noise_input, real_seq[:, :-1])
        fake_seq = torch.argmax(fake_output_logits, dim=-1)
        pad_token_id = token_to_int['<pad>']
        fake_seq_padded = F.pad(fake_seq, (0, 1), "constant", pad_token_id)
        real_seq_embedded = discriminator.embedding(real_seq)
        fake_seq_embedded = discriminator.embedding(fake_seq_padded.detach())
        real_validity = discriminator(real_seq_embedded)
        fake_validity = discriminator(fake_seq_embedded)
        gradient_penalty = compute_gradient_penalty(discriminator, real_seq_embedded.detach(), fake_seq_embedded.detach())
        d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + LAMBDA_GP * gradient_penalty
        d_loss.backward()
        optimizer_d.step()
        # -----------------
        #  Train Generator
        # -----------------
        if i % 5 == 0:
            optimizer_g.zero_grad()
            noise_input_g = torch.randint(0, vocab_size, real_seq.shape, device=device)
            fake_output_logits_g = generator(noise_input_g, real_seq[:, :-1])

            # ======================= [START OF CORRECTION] =======================
            # The original code used argmax here, which is not differentiable.
            # We replace it with a differentiable approximation (soft-embedding).

            # 1. Get probabilities from logits
            # Shape: (batch_size, seq_len-1, vocab_size)
            fake_probs_g = F.softmax(fake_output_logits_g, dim=-1)

            # 2. Get the discriminator's embedding matrix
            # Shape: (vocab_size, d_model)
            embedding_matrix = discriminator.embedding.weight

            # 3. Calculate the "soft" embedding by multiplying probabilities
            #    with the embedding matrix. This is a differentiable
            #    replacement for argmax -> embedding.
            #    (B, L, V) @ (V, D) -> (B, L, D) where L = MAX_LEN - 1
            fake_seq_g_embedded_soft = torch.matmul(fake_probs_g, embedding_matrix)

            # 4. Pad the soft sequence to be MAX_LEN, just as the argmax'd one was.
            #    We pad the sequence dimension (dim 1) by 1 at the end.
            #    Pad format: (last_dim_left, last_dim_right, 2nd_last_dim_left, 2nd_last_dim_right, ...)
            fake_seq_g_embedded_soft_padded = F.pad(fake_seq_g_embedded_soft, (0, 0, 0, 1), "constant", 0)

            # 5. Calculate the adversarial loss using this soft, differentiable embedding
            #    The discriminator's forward pass accepts embedded sequences.
            adv_loss = -torch.mean(discriminator(fake_seq_g_embedded_soft_padded))
            # ======================== [END OF CORRECTION] ========================

            # The supervised loss remains the same, helping the generator
            # learn the basic structure of SELFIES.
            sup_loss = criterion_g(fake_output_logits_g.view(-1, vocab_size), real_seq[:, 1:].reshape(-1))

            # The total loss is now correctly composed of both
            # adversarial and supervised components.
            g_loss = adv_loss + sup_loss

            g_loss.backward()
            optimizer_g.step()

    # Print losses at the end of the epoch
    if 'd_loss' in locals() and 'g_loss' in locals():
      print(f"[Epoch {epoch+1}/{N_EPOCHS}] [D loss: {d_loss.item():.4f}] [G loss: {g_loss.item():.4f}]")

    # --- 5. Save Checkpoint After Each Epoch ---
    print(f"💾 Saving WGAN checkpoint for epoch {epoch}...")
    torch.save({
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'discriminator_state_dict': discriminator.state_dict(),
        'optimizer_g_state_dict': optimizer_g.state_dict(),
        'optimizer_d_state_dict': optimizer_d.state_dict(),
    }, wgan_checkpoint_path)
    print("Checkpoint saved successfully to Google Drive.")

✅ WGAN checkpoint found! Loading state from: /content/drive/My Drive/molgen_project_(1)/wgan_checkpoint.pth
Resuming training from epoch 50
Starting WGAN-GP Training...


In [ ]:
import pubchempy as pcp
from rdkit import Chem
import selfies as sf

def get_smiles_universal(disease_name):
    """Fetches high-quality SMILES with a fallback if ChEMBL is down."""
    # --- 1. TRY CHEMBL (EBI) ---
    try:
        from chembl_webresource_client.new_client import new_client
        indications = new_client.drug_indication.filter(mesh_heading__icontains=disease_name)
        if indications and len(indications) > 0:
            chembl_id = indications[0]['molecule_chembl_id']
            mol = new_client.molecule.filter(molecule_chembl_id=chembl_id).only(['molecule_structures'])
            smi = mol[0]['molecule_structures']['canonical_smiles']
            print(f"✅ Found via ChEMBL: {smi}")
            return smi
    except Exception as e:
        print(f"⚠️ ChEMBL API Error (Server might be down): {e}")

    # --- 2. FALLBACK TO PUBCHEM ---
    try:
        print(f"🔍 Searching PubChem for: {disease_name}...")
        results = pcp.get_compounds(disease_name, 'name')
        if results and len(results) > 0:
            smi = results[0].canonical_smiles
            print(f"✅ Found via PubChem: {smi}")
            return smi
    except Exception as e:
        print(f"❌ PubChem Error: {e}")

    # --- 3. ABSOLUTE FALLBACK (Common Lead) ---
    print("🆘 Both APIs failed. Using fallback lead scaffold.")
    return "Nc1ccnc2cccc(Cl)c12" # Chloroquine core

In [ ]:
# ==============================================================================
# --- PHASE 1: WGAN-GP FINE TUNING ---
# ==============================================================================
import torch.nn.functional as F

print(f"\n🔬 Fine-tuning WGAN on Lead Scaffold: {final_smi}")

generator.train()
discriminator.train()

# We use 3 epochs to "prime" the model without causing severe Mode Collapse
FINE_TUNE_EPOCHS = 3

for epoch in range(FINE_TUNE_EPOCHS):
    for i, real_seq in enumerate(fine_tune_loader):
        real_seq = real_seq.to(device)

        # --- 1. Train Discriminator ---
        optimizer_d.zero_grad()
        noise_input = torch.randint(0, vocab_size, real_seq.shape, device=device)
        fake_output_logits = generator(noise_input, real_seq[:, :-1])
        fake_seq = torch.argmax(fake_output_logits, dim=-1)

        fake_seq_padded = F.pad(fake_seq, (0, 1), "constant", token_to_int['<pad>'])
        real_seq_embedded = discriminator.embedding(real_seq)
        fake_seq_embedded = discriminator.embedding(fake_seq_padded.detach())

        real_validity = discriminator(real_seq_embedded)
        fake_validity = discriminator(fake_seq_embedded)
        gradient_penalty = compute_gradient_penalty(discriminator, real_seq_embedded.detach(), fake_seq_embedded.detach())

        d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + LAMBDA_GP * gradient_penalty
        d_loss.backward()
        optimizer_d.step()

        # --- 2. Train Generator (using differentiable soft-embedding) ---
        if i % 5 == 0:
            optimizer_g.zero_grad()
            noise_input_g = torch.randint(0, vocab_size, real_seq.shape, device=device)
            fake_output_logits_g = generator(noise_input_g, real_seq[:, :-1])

            fake_probs_g = F.softmax(fake_output_logits_g, dim=-1)
            embedding_matrix = discriminator.embedding.weight
            fake_seq_g_embedded_soft = torch.matmul(fake_probs_g, embedding_matrix)
            fake_seq_g_embedded_soft_padded = F.pad(fake_seq_g_embedded_soft, (0, 0, 0, 1), "constant", 0)

            adv_loss = -torch.mean(discriminator(fake_seq_g_embedded_soft_padded))
            sup_loss = criterion_g(fake_output_logits_g.view(-1, vocab_size), real_seq[:, 1:].reshape(-1))
            g_loss = adv_loss + sup_loss
            g_loss.backward()
            optimizer_g.step()

    print(f"✅ Fine-tune Epoch {epoch+1}/{FINE_TUNE_EPOCHS} complete | G Loss: {g_loss.item():.4f}")


🔬 Fine-tuning WGAN on Lead Scaffold: CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(C(=O)N[C@@H](CCC(=O)O)C(=O)O)cc1


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


✅ Fine-tune Epoch 1/3 complete | G Loss: -263.4641
✅ Fine-tune Epoch 2/3 complete | G Loss: -304.7412
✅ Fine-tune Epoch 3/3 complete | G Loss: -323.3462


In [ ]:
# ==============================================================================
# --- PHASE 2: INTERACTIVE WEIGHTS & REWARD AUDITOR ---
# ==============================================================================
from rdkit import Chem, DataStructs
from rdkit.Chem import QED, Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

# 1. Setup Global Fingerprinter & Scaffold
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
seed_mol = Chem.MolFromSmiles(final_smi)
seed_scaffold = MurckoScaffold.GetScaffoldForMol(seed_mol)
scaffold_smiles = Chem.MolToSmiles(seed_scaffold)

# Fetch Reference Drugs for Disease Similarity
def get_reference_fps(disease):
    fps = []
    try:
        inds = new_client.drug_indication.filter(mesh_heading__icontains=disease)
        for ind in inds[:5]:
            mol_data = new_client.molecule.filter(molecule_chembl_id=ind['molecule_chembl_id']).only(['molecule_structures'])
            if mol_data and mol_data[0]['molecule_structures']:
                m = Chem.MolFromSmiles(mol_data[0]['molecule_structures']['canonical_smiles'])
                if m: fps.append(morgan_gen.GetFingerprint(m))
    except: pass
    return fps
reference_fps = get_reference_fps(best_disease)

# 2. Interactive User Weights
user_weights = {}
print(f"\n📊 Define Optimization Goals for {best_disease.upper()}")
print("(Press Enter to keep defaults: QED: 0.8, logP: -0.2, SA: 0.5, Tox: -2.0)")

def get_user_weight(prompt, default):
    val = input(f"{prompt} (default {default}): ")
    return float(val) if val else default

user_weights['qed'] = get_user_weight("QED weight", 0.8)
user_weights['logp'] = get_user_weight("logP weight", -0.2)
user_weights['sa_score'] = get_user_weight("Synthesizability weight", 0.5)
user_weights['toxicity'] = get_user_weight("Toxicity Penalty", -2.0)

# 3. The Strict Reward Function
def unified_reward_function(selfies_string, weights):
    try:
        smi = sf.decoder(selfies_string)
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return -30.0

        # Anti-Snake & Anti-Hallucination Penalties
        stability_penalty = 0.0
        if "CCCCCCC" in smi: stability_penalty -= 80.0
        if "NNN" in smi: stability_penalty -= 60.0
        if Descriptors.MolWt(mol) > 600: stability_penalty -= 40.0

        # Disease Relevance Similarity
        relevance_reward = 0.0
        if reference_fps:
            gen_fp = morgan_gen.GetFingerprint(mol)
            sims = [DataStructs.TanimotoSimilarity(gen_fp, ref_fp) for ref_fp in reference_fps]
            relevance_reward = max(sims) * 50.0 if sims else 0.0

        # Features & Scaffold
        func_bonus = 10.0 if ("O" in smi or "N" in smi) else -20.0
        scaffold_reward = 30.0 if mol.HasSubstructMatch(seed_scaffold) else -50.0

        qed_score = QED.qed(mol)
        return (weights['qed'] * qed_score * 15.0) + relevance_reward + scaffold_reward + stability_penalty + func_bonus
    except:
        return -30.0


📊 Define Optimization Goals for MALARIA
(Press Enter to keep defaults: QED: 0.8, logP: -0.2, SA: 0.5, Tox: -2.0)
QED weight (default 0.8): 0.8
logP weight (default -0.2): -0.2
Synthesizability weight (default 0.5): 0.5
Toxicity Penalty (default -2.0): -2.0


In [ ]:
# ==============================================================================
# --- PHASE 3: REINFORCEMENT LEARNING & GENERATION ---
# ==============================================================================
import torch, gc
import numpy as np
from torch.optim.lr_scheduler import StepLR

print("\n🧠 Starting Reinforcement Learning Phase (Aligning with your weights)...")
optimizer_g_rl = torch.optim.Adam(generator.parameters(), lr=1e-5)
scheduler = StepLR(optimizer_g_rl, step_size=20, gamma=0.7)

# RL Loop
for step in range(50):
    generator.train()
    optimizer_g_rl.zero_grad()
    generated_selfies = []

    with torch.no_grad():
        input_seq = torch.full((BATCH_SIZE, 1), token_to_int['<sos>'], device=device, dtype=torch.long)
        for _ in range(MAX_LEN - 1):
            dummy_src = torch.zeros_like(input_seq)
            probs = F.softmax(generator(dummy_src, input_seq)[:, -1, :] / 0.8, dim=-1)
            input_seq = torch.cat([input_seq, torch.multinomial(probs, 1)], dim=1)

    for i in range(BATCH_SIZE):
        seq = input_seq[i, 1:].tolist()
        try: seq = seq[:seq.index(token_to_int['<eos>'])]
        except: pass
        generated_selfies.append("".join([int_to_token[tid] for tid in seq if tid in int_to_token]))

    batch_rewards = [unified_reward_function(s, user_weights) for s in generated_selfies]
    rewards_tensor = torch.tensor(batch_rewards, device=device, dtype=torch.float32)
    normalized_rewards = (rewards_tensor - rewards_tensor.mean()) / (rewards_tensor.std() + 1e-6)

    for i in range(BATCH_SIZE):
        if not generated_selfies[i]: continue
        try:
            tokens = ['<sos>'] + list(sf.split_selfies(generated_selfies[i]))
            token_ids = [token_to_int.get(tok, 0) for tok in tokens]
            if len(token_ids) <= 1: continue
            seq_tensor = torch.tensor([token_ids], device=device, dtype=torch.long)
            noise_src = torch.zeros(seq_tensor[:, :-1].shape, device=device, dtype=torch.long)
            log_probs = F.log_softmax(generator(noise_src, seq_tensor[:, :-1]), dim=-1)
            gen_log_probs = log_probs.squeeze(0).gather(1, seq_tensor[:, 1:].squeeze(0).unsqueeze(1))

            if gen_log_probs.nelement() > 0:
                policy_loss = -gen_log_probs.sum() * normalized_rewards[i] / BATCH_SIZE
                policy_loss.backward()
        except: continue

    torch.nn.utils.clip_grad_norm_(generator.parameters(), 0.5)
    optimizer_g_rl.step()
    scheduler.step()
    if (step + 1) % 10 == 0:
        print(f"  [RL Step {step+1}/50] Avg Reward: {np.mean(batch_rewards):.2f}")

gc.collect()
torch.cuda.empty_cache()

# Fast Generation
def fast_generate_optimized_leads(generator, prefix_smi, num_samples=200):
    generator.eval()
    print(f"\n🏭 Generating optimized candidates for {best_disease.upper()}...")
    try:
        prefix_ids = [token_to_int.get(t, token_to_int['<pad>']) for t in list(sf.split_selfies(sf.encoder(scaffold_smiles)))]
    except: prefix_ids = []

    results_list, seen = [], set()

    with torch.no_grad():
        for _ in range(num_samples):
            seq = ([token_to_int['<sos>']] + prefix_ids)[:MAX_LEN - 1]
            for _ in range(MAX_LEN - len(seq)):
                seq_t = torch.tensor([seq], device=device, dtype=torch.long)
                noise = torch.zeros(seq_t.shape, device=device, dtype=torch.long)
                probs = F.softmax(generator(noise, seq_t)[:, -1, :] / 0.4, dim=-1)
                next_tok = torch.multinomial(probs, 1).item()
                seq.append(next_tok)
                if next_tok == token_to_int['<eos>']: break

            try:
                final_ids = seq[1:seq.index(token_to_int['<eos>'])] if token_to_int['<eos>'] in seq else seq[1:]
                selfies_out = "".join([int_to_token.get(idx, '') for idx in final_ids])
                smi = sf.decoder(selfies_out)

                if smi not in seen:
                    reward = unified_reward_function(selfies_out, user_weights)
                    if Chem.MolFromSmiles(smi).HasSubstructMatch(seed_scaffold) and reward > 5.0:
                        results_list.append((reward, smi))
                        seen.add(smi)
            except: continue

    results_list.sort(key=lambda x: x[0], reverse=True)
    return results_list[:10]

results = fast_generate_optimized_leads(generator, scaffold_smiles)
print(f"✅ Pipeline Complete! Found {len(results)} high-quality candidates.")


🧠 Starting Reinforcement Learning Phase (Aligning with your weights)...
  [RL Step 10/50] Avg Reward: -54.24
  [RL Step 20/50] Avg Reward: -57.57
  [RL Step 30/50] Avg Reward: -60.42
  [RL Step 40/50] Avg Reward: -69.11
  [RL Step 50/50] Avg Reward: -58.19

🏭 Generating optimized candidates for MALARIA...
✅ Pipeline Complete! Found 10 high-quality candidates.


In [ ]:
results

[(56.744805749837646, 'C1=CC=C(NCC2=CN=C3N=CN=CC3=N2)C=C1C=O'),
 (55.56998459921622, 'C1=CC=C(N2CC3=CN=C4N=CN=CC4=N3)C=C12'),
 (55.42559330415884, 'C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CC'),
 (54.91697312970078, 'C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=C'),
 (54.73748907874649, 'C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CC=CCC'),
 (54.64185608601997, 'C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CC=C'),
 (54.433327047962266, 'C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2N=CC=CCN=C'),
 (54.39554163847278, 'C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2(C)CNN=O'),
 (54.36052186554882, 'C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CCN=O'),
 (54.203895439632866, 'C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CC=CC=CC')]

In [ ]:
#Cell 13: The Medicinal Chemistry Validation Filter
# Cell 13: Medicinal Chemistry Validation Filter
from rdkit.Chem import Descriptors, Lipinski, QED, rdMolDescriptors
import pandas as pd

def medicinal_chemistry_audit(results_list):
    print("--- 🧪 Starting Pharmaceutical Lead Audit ---")
    final_report = []

    for reward, smi in results_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: continue

        # 1. Physical Properties (Lipinski's Rule of Five)
        mw = Descriptors.MolWt(mol)           # Molecular Weight (< 500)
        logp = Descriptors.MolLogP(mol)       # Lipophilicity (< 5)
        hbd = Lipinski.NumHDonors(mol)        # H-Bond Donors (<= 5)
        hba = Lipinski.NumHAcceptors(mol)     # H-Bond Acceptors (<= 10)

        # 2. Stability Filter (The "Allene" Check)
        is_stable = "C=C=C" not in smi and "C#C#C" not in smi

        # 3. Drug-Likeness (QED)
        qed_score = QED.qed(mol)

        # 4. Status Determination
        passes_lipinski = (mw < 500 and logp < 5 and hbd <= 5 and hba <= 10)
        status = "✅ PASS" if (passes_lipinski and is_stable) else "❌ FAIL"

        reason = "Optimal Lead" if status == "✅ PASS" else ""
        if not is_stable: reason += "Unstable Polyene; "
        if not passes_lipinski: reason += "Lipinski Violation; "
        if qed_score < 0.3: reason += "Low QED; "

        final_report.append({
            "SMILES": smi,
            "MW": round(mw, 2),
            "LogP": round(logp, 2),
            "QED": round(qed_score, 3),
            "Status": status,
            "Reason": reason
        })

    return pd.DataFrame(final_report)

# Execute the audit using 'results' from Cell 12
df_results = medicinal_chemistry_audit(results)

df_results.rename(columns={'SMILES': 'SELFIES'}, inplace=True, errors='ignore')

display(df_results)
df_results.to_csv("malaria_leads.csv", index=False)

--- 🧪 Starting Pharmaceutical Lead Audit ---


,SELFIES,MW,LogP,QED,Status,Reason
0,C1=CC=C(NCC2=CN=C3N=CN=CC3=N2)C=C1C=O,265.28,1.84,0.725,✅ PASS,Optimal Lead
1,C1=CC=C(N2CC3=CN=C4N=CN=CC4=N3)C=C12,235.25,2.07,0.680,✅ PASS,Optimal Lead
2,C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CC,275.32,2.80,0.623,✅ PASS,Optimal Lead
3,C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=C,261.29,2.41,0.613,✅ PASS,Optimal Lead
4,C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CC=CCC,315.38,3.74,0.608,✅ PASS,Optimal Lead
5,C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CC=C,287.33,2.96,0.625,✅ PASS,Optimal Lead
6,C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2N=CC=CCN=C,343.39,2.76,0.666,✅ PASS,Optimal Lead
7,C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2(C)CNN=O,321.34,1.92,0.528,✅ PASS,Optimal Lead
8,C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CCN=O,304.31,2.54,0.569,✅ PASS,Optimal Lead
9,C12=CC=C(NCC3=CN=C4N=CN=CC4=N3)C=C1C2=CC=CC=CC,327.39,3.91,0.563,✅ PASS,Optimal Lead


In [ ]:
# Cell 14: 3D Conformation & Force Field Minimization
from rdkit.Chem import AllChem

# Pick the top molecule that passed the audit
passing_leads = df_results[df_results['Status'] == "✅ PASS"]
if not passing_leads.empty:
    best_smi = passing_leads.iloc[0]['SELFIES']
    best_mol = Chem.MolFromSmiles(best_smi)
    best_mol = Chem.AddHs(best_mol) # Add hydrogens for realistic 3D

    # Generate 3D Coordinates
    AllChem.EmbedMolecule(best_mol, AllChem.ETKDG())
    # Minimize energy using MMFF94 force field
    AllChem.MMFFOptimizeMolecule(best_mol)
    print(f"3D Structure generated and energy-minimized for: {best_smi[:40]}...")
else:
    print("No leads passed the audit. Try adjusting weights in Cell 11.")

3D Structure generated and energy-minimized for: C1=CC=C(NCC2=CN=C3N=CN=CC3=N2)C=C1C=O...


In [ ]:
!pip install py3Dmol

In [ ]:
# Cell 15: Interactive 3D Molecular Visualization
import py3Dmol

def visualize_3d(mol):
    mblock = Chem.MolToMolBlock(mol)
    viewer = py3Dmol.view(width=600, height=400)
    viewer.addModel(mblock, 'mol')
    viewer.setStyle({'stick': {}, 'sphere': {'scale': 0.3}})
    viewer.setBackgroundColor('white')
    viewer.zoomTo()
    return viewer.show()

if 'best_mol' in locals():
    visualize_3d(best_mol)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.